Objetivo

En esta etapa se entrenarán y compararán distintos algoritmos de clasificación para predecir la variable smoking. Se utilizará como métrica principal el F1-score para la clase positiva (smoking = 1), dado que fue la métrica definida en la consigna y resulta adecuada para evaluar simultáneamente precisión y exhaustividad.

Como se concluyó en la etapa de EDA, se conservarán todas las variables predictoras disponibles, ya que no se identificaron atributos claramente irrelevantes y el número total de variables es reducido.

Secciones:

Cargar el dataset preprocesado.
Separar X e y.
Hacer train_test_split.
Entrenar varios modelos.
Compararlos usando F1-score.
Elegir el mejor modelo.
Ajustar hiperparámetros del modelo ganador.
Guardar el modelo final.

1.Carga del dataset preprocesado

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/smoking_prediction_preprocessed.csv"
)

df.head()

,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,tartar,smoking
0,0,40,155,60,3.38,0.04,0.04,0.04,0.04,4.75,...,5.25,0.51,0.04,0.00,0.75,0.79,1.13,0,1,0
1,0,40,160,60,3.38,0.01,0.00,0.04,0.04,4.96,...,5.29,0.50,0.04,0.00,0.92,0.79,0.75,0,1,0
2,1,55,170,60,3.33,0.01,0.01,0.04,0.04,5.75,...,6.29,0.63,0.04,0.04,0.88,0.67,0.92,0,0,1
3,1,40,165,70,3.67,0.05,0.05,0.04,0.04,4.17,...,9.42,0.59,0.04,0.04,0.79,1.08,0.75,0,1,0
4,0,40,155,60,3.58,0.04,0.04,0.04,0.04,5.00,...,4.46,0.50,0.04,0.00,0.67,0.58,0.92,0,0,0


2.Separación X e y

In [2]:
X = df.drop(columns=["smoking"])

y = df["smoking"]

print(X.shape)
print(y.shape)

(50000, 24)
(50000,)


3.Hacemos train_test_split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
#Verificación

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

smoking
0    0.633425
1    0.366575
Name: proportion, dtype: float64
smoking
0    0.6334
1    0.3666
Name: proportion, dtype: float64


4. Primer modelo base - Regresión Logistica

In [5]:
#Imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    f1_score
)

In [6]:
#Training

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train, y_train)

D:\Gaby\anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [7]:
#Predict

y_pred_lr = lr.predict(X_test)

In [8]:
#Metrics

print(classification_report(
    y_test,
    y_pred_lr
))

              precision    recall  f1-score   support

           0       0.81      0.77      0.79      6334
           1       0.63      0.70      0.67      3666

    accuracy                           0.74     10000
   macro avg       0.72      0.73      0.73     10000
weighted avg       0.75      0.74      0.74     10000



In [9]:
f1_lr = f1_score(
    y_test,
    y_pred_lr
)

print(f"F1-score: {f1_lr:.4f}")

F1-score: 0.6651


Observación:

Se obtuvo un F1-score de 0.6651 para la clase positiva. Durante el entrenamiento se observó una advertencia de convergencia, lo que sugiere que el modelo podría beneficiarse de un escalado previo de las variables.

5. Primer modelo base - Regresión Logistica + Standard Scaler

In [10]:
#Imports
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [11]:
pipeline_lr_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [12]:
#Training

pipeline_lr_scaled.fit(
    X_train,
    y_train
)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [13]:
#Predict

y_pred_lr_scaled = pipeline_lr_scaled.predict(
    X_test
)

In [14]:
#Metrics

print(classification_report(
    y_test,
    y_pred_lr_scaled
))

              precision    recall  f1-score   support

           0       0.81      0.77      0.79      6334
           1       0.64      0.70      0.67      3666

    accuracy                           0.74     10000
   macro avg       0.73      0.73      0.73     10000
weighted avg       0.75      0.74      0.75     10000



In [15]:
f1_lr_scaled = f1_score(
    y_test,
    y_pred_lr_scaled
)

print(f"F1-score: {f1_lr_scaled:.4f}")

F1-score: 0.6650


Conclusion:
Se evaluó una segunda versión del modelo Logistic Regression incorporando un escalado de variables mediante StandardScaler. Aunque el escalado eliminó los problemas de convergencia observados inicialmente, el F1-score obtenido fue prácticamente idéntico al del modelo base. Esto sugiere que el rendimiento del algoritmo lineal se encuentra cerca de su límite para este problema y motiva la evaluación de modelos más complejos.

6. Random Forest

In [16]:
#Imports

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

In [17]:
#Training

rf = RandomForestClassifier(
    n_estimators=100, #100 arboles
    random_state=42,
    n_jobs=-1 #utiliza todos los nucleos del sistema para más performance
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
#Predict

y_pred_rf = rf.predict(X_test)

In [19]:
#Metrics

print(classification_report(
    y_test,
    y_pred_rf
))

              precision    recall  f1-score   support

           0       0.85      0.83      0.84      6334
           1       0.72      0.75      0.73      3666

    accuracy                           0.80     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.80      0.80      0.80     10000



In [20]:
f1_rf = f1_score(
    y_test,
    y_pred_rf
)

print(f"F1-score: {f1_rf:.4f}")

F1-score: 0.7346


In [22]:
#Importancia de Variables para confirmación de lo visto en el EDA.

importancias = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importancias.head(10)

gender                 0.110715
Gtp                    0.090452
height(cm)             0.071811
triglyceride           0.070758
hemoglobin             0.060515
LDL                    0.051910
Cholesterol            0.050034
HDL                    0.049823
ALT                    0.049470
fasting blood sugar    0.048757
dtype: float64

Conclusion: El modelo Random Forest obtuvo un F1-score de 0.7346 sobre la clase positiva, superando ampliamente a los modelos de regresión logística evaluados previamente. El análisis de importancia de variables mostró que atributos como género, Gtp, altura, triglicéridos y hemoglobina tienen una influencia significativa en la predicción del hábito de fumar, resultados consistentes con los hallazgos obtenidos durante el análisis exploratorio de datos.

7. Random Forest con optimización GridSearchCV

In [23]:
#Imports

from sklearn.model_selection import GridSearchCV

In [24]:
#Combinations

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

In [25]:
#GridSearchCv config

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [26]:
#Training

grid_rf.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [10, 20, ...], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [28]:
#Best combination found

print(grid_rf.best_params_)

{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}


In [34]:
#Cross Score found 
print(grid_rf.best_score_)

0.7216599633917652


In [35]:
#Elegimos el mejor arbol
best_rf = grid_rf.best_estimator_

In [36]:
#Usamos los parametros del mejor arbol
y_pred_best_rf = best_rf.predict(X_test)

In [37]:
#Best tree report

print(classification_report(
    y_test,
    y_pred_best_rf
))

f1_best_rf = f1_score(
    y_test,
    y_pred_best_rf
)

print(f"F1-score: {f1_best_rf:.4f}")

              precision    recall  f1-score   support

           0       0.85      0.83      0.84      6334
           1       0.71      0.76      0.74      3666

    accuracy                           0.80     10000
   macro avg       0.78      0.79      0.79     10000
weighted avg       0.80      0.80      0.80     10000

F1-score: 0.7356


Conclusion: Se evaluaron distintos modelos de clasificación utilizando como métrica principal el F1-score para la clase positiva (smoking = 1). Los modelos de Regresión Logística obtuvieron resultados similares, con un F1-score cercano a 0.67. Posteriormente se entrenó un Random Forest, que incrementó significativamente el rendimiento hasta un F1-score de 0.7346. Finalmente, se realizó una optimización de hiperparámetros mediante GridSearchCV, obteniendo una mejora adicional y alcanzando un F1-score de 0.7356. Por este motivo, se selecciona el Random Forest optimizado como modelo candidato para las etapas de validación y generación de predicciones.

6. XGBoost

In [41]:
#Imports and Config

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

In [42]:
#Training

xgb.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [43]:
#Predict

y_pred_xgb = xgb.predict(X_test)

In [44]:
#Metrics

print(classification_report(
    y_test,
    y_pred_xgb
))

f1_xgb = f1_score(
    y_test,
    y_pred_xgb
)

print(f"F1-score: {f1_xgb:.4f}")

              precision    recall  f1-score   support

           0       0.83      0.80      0.82      6334
           1       0.68      0.72      0.70      3666

    accuracy                           0.77     10000
   macro avg       0.75      0.76      0.76     10000
weighted avg       0.77      0.77      0.77     10000

F1-score: 0.6958


Conclusión FINAL:

También se evaluó un modelo XGBoost, ampliamente utilizado en problemas de clasificación sobre datos tabulares. Sin embargo, con la configuración inicial ensayada obtuvo un F1-score de 0.6958, inferior al alcanzado por Random Forest. Por este motivo se decidió continuar con el modelo Random Forest optimizado como la mejor alternativa de todas las evaluadas.